<a href="https://colab.research.google.com/github/sharvani1357/Positional_Encoding/blob/main/Resume_Positional_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# TASK 1 : RESUME DATASET ANALYSIS (EDA)
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

# ------------------------------------------
# Load Dataset
# ------------------------------------------
df = pd.read_csv("../Datasets/Resume.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns)

# ------------------------------------------
# Use Resume Text Column
# ------------------------------------------
resume_col = None

for col in df.columns:
    if "resume" in col.lower():
        resume_col = col
        break

print("\nResume Text Column:", resume_col)

# ------------------------------------------
# Number of Resumes
# ------------------------------------------
num_resumes = len(df)
print("\nTotal Resumes:", num_resumes)

# ------------------------------------------
# Categories
# ------------------------------------------
if "Category" in df.columns:
    print("\nUnique Categories:", df["Category"].nunique())
    print("\nCategory Counts:")
    print(df["Category"].value_counts())

# ------------------------------------------
# Average Resume Length
# ------------------------------------------
df["Resume_Length"] = df[resume_col].astype(str).apply(
    lambda x: len(x.split())
)

avg_length = df["Resume_Length"].mean()

print("\nAverage Resume Length:", round(avg_length, 2), "words")

# ------------------------------------------
# Extract Skills
# ------------------------------------------

skills_list = [

    # Programming
    "python","java","c","c++","javascript","typescript",
    "html","css","php","r","sql",

    # Data Science
    "machine learning","deep learning","artificial intelligence",
    "data science","nlp","computer vision",

    # Libraries
    "numpy","pandas","matplotlib","seaborn",
    "scikit-learn","tensorflow","keras","pytorch",

    # Databases
    "mysql","mongodb","postgresql",

    # Cloud
    "aws","azure","gcp","docker","kubernetes",

    # Soft Skills
    "communication","leadership","teamwork",
    "problem solving","analytical skills",

    # Business
    "marketing","accounting","customer service",
    "human resources","project management"
]

skill_counter = Counter()

for resume in df[resume_col].astype(str):

    text = resume.lower()

    for skill in skills_list:
        if skill in text:
            skill_counter[skill] += 1

# ------------------------------------------
# Most Common Skills
# ------------------------------------------
print("\nTop 20 Skills:\n")

for skill, count in skill_counter.most_common(20):
    print(f"{skill}: {count}")

# ==========================================
# VISUALIZATION 1
# Skill Frequency Chart
# ==========================================

top_skills = skill_counter.most_common(15)

skills = [x[0] for x in top_skills]
counts = [x[1] for x in top_skills]

plt.figure(figsize=(12,6))
plt.bar(skills, counts)
plt.xticks(rotation=45)
plt.title("Top Skills in Resume Dataset")
plt.xlabel("Skills")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ==========================================
# VISUALIZATION 2
# Resume Category Distribution
# ==========================================

if "Category" in df.columns:

    plt.figure(figsize=(12,6))

    category_counts = df["Category"].value_counts()

    plt.bar(category_counts.index,
            category_counts.values)

    plt.xticks(rotation=90)
    plt.title("Resume Category Distribution")
    plt.xlabel("Category")
    plt.ylabel("Number of Resumes")
    plt.tight_layout()
    plt.show()

# ==========================================
# SUMMARY
# ==========================================

print("\n========== EDA SUMMARY ==========")
print("Total Resumes:", num_resumes)

if "Category" in df.columns:
    print("Number of Categories:",
          df["Category"].nunique())

print("Average Resume Length:",
      round(avg_length,2),"words")

print("\nTop 10 Skills:")

for skill,count in skill_counter.most_common(10):
    print(skill,"->",count)

FileNotFoundError: [Errno 2] No such file or directory: 'Resume.csv'

Task 2: Text Preprocessing
Perform:
Lowercase
Remove Special Characters
Tokenization
Padding
Vocabulary Creation

In [ ]:
# ==========================================
# TASK 2 : TEXT PREPROCESSING
# ==========================================

import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------
# Load Dataset
# ------------------------------------------

df = pd.read_csv("Resume.csv")

# ------------------------------------------
# Detect Resume Column
# ------------------------------------------

resume_col = None

for col in df.columns:
    if "resume" in col.lower():
        resume_col = col
        break

print("Resume Column:", resume_col)

# ------------------------------------------
# Lowercase + Remove Special Characters
# ------------------------------------------

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df["clean_resume"] = df[resume_col].apply(clean_text)

print("\nSample Cleaned Resume:\n")
print(df["clean_resume"].iloc[0][:500])

# ==========================================
# TOKENIZATION
# ==========================================

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    df["clean_resume"]
)

# Convert text into integer sequences
sequences = tokenizer.texts_to_sequences(
    df["clean_resume"]
)

print("\nSample Tokenized Sequence:\n")
print(sequences[0][:50])

# ==========================================
# VOCABULARY CREATION
# ==========================================

word_index = tokenizer.word_index

vocab_size = len(word_index) + 1

print("\nVocabulary Size:", vocab_size)

print("\nFirst 20 Vocabulary Words:\n")

for i, (word, idx) in enumerate(word_index.items()):

    print(word, ":", idx)

    if i == 19:
        break

# ==========================================
# PADDING
# ==========================================

max_length = max(
    len(seq) for seq in sequences
)

print("\nMaximum Sequence Length:", max_length)

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("\nPadded Shape:")
print(padded_sequences.shape)

print("\nSample Padded Sequence:\n")
print(padded_sequences[0][:100])

# ==========================================
# CONVERT TO TENSOR
# ==========================================

X = tf.constant(
    padded_sequences,
    dtype=tf.int32
)

print("\nTensor Shape:")
print(X.shape)

# ==========================================
# SUMMARY
# ==========================================

print("\n========== PREPROCESSING SUMMARY ==========")

print("Total Resumes:",
      len(df))

print("Vocabulary Size:",
      vocab_size)

print("Maximum Sequence Length:",
      max_length)

print("Tensor Shape:",
      X.shape)

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    Dense,
    GlobalAveragePooling1D,
    Dropout
)
from tensorflow.keras.models import Model

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load Dataset

df = pd.read_csv("Resume.csv")

# Detect Resume Column

resume_col = None

for col in df.columns:
    if "resume" in col.lower():
        resume_col = col
        break

print("Resume Column:", resume_col)

# Text Cleaning

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df["clean_resume"] = df[resume_col].apply(clean_text)

# Tokenization

tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(df["clean_resume"])

sequences = tokenizer.texts_to_sequences(
    df["clean_resume"]
)

# Vocabulary

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)

# Check Sequence Lengths

lengths = [len(seq) for seq in sequences]

print("Maximum Length :", max(lengths))
print("Average Length :", int(np.mean(lengths)))

# Padding (Reduced to avoid OOM)

max_length = 300

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Input Shape:", X.shape)

# Label Encoding

encoder = LabelEncoder()

y = encoder.fit_transform(df["Category"])

num_classes = len(np.unique(y))

print("Number of Classes:", num_classes)

# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model Parameters

embedding_dim = 64
num_heads = 2

# Input Layer

inputs = Input(shape=(max_length,))

# Embedding Layer

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)(inputs)

# Multi Head Self Attention

attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=32
)(
    query=embedding_layer,
    value=embedding_layer,
    key=embedding_layer
)

# Pooling

pooling = GlobalAveragePooling1D()(attention_output)

# Dense Layer

dense = Dense(
    64,
    activation='relu'
)(pooling)

drop = Dropout(0.3)(dense)

# Output Layer

outputs = Dense(
    num_classes,
    activation='softmax'
)(drop)

# Build Model

baseline_model = Model(
    inputs=inputs,
    outputs=outputs
)

baseline_model.summary()

# Compile Model

baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training

start_time = time.time()

history = baseline_model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=8,
    verbose=1
)

end_time = time.time()

training_time = end_time - start_time

# Evaluation

loss, accuracy = baseline_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

# Results

print("\nBaseline Model Results")
print("Accuracy      :", round(accuracy * 100, 2), "%")
print("Loss          :", round(loss, 4))
print("Training Time :", round(training_time, 2), "seconds")

# Store Metrics

baseline_accuracy = accuracy
baseline_loss = loss
baseline_training_time = training_time

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Positional Encoding Function

def positional_encoding(position, d_model):

    positions = np.arange(position)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])

    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(PE, dtype=tf.float32)

# Parameters

max_position = 50
d_model = 64

# Generate Positional Encoding

PE = positional_encoding(
    max_position,
    d_model
)

print("Positional Encoding Shape:")
print(PE.shape)

print("\nFirst Position Encoding Vector:")
print(PE[0].numpy())

# Visualization

plt.figure(figsize=(10,6))

plt.imshow(
    PE.numpy(),
    aspect='auto'
)

plt.colorbar()

plt.xlabel("Embedding Dimensions")

plt.ylabel("Word Positions")

plt.title("Positional Encoding Matrix")

plt.show()

In [ ]:
# Example Embeddings

word_embeddings = tf.random.normal(
    shape=(1, max_position, d_model)
)

# Add Positional Encoding

position_aware_embeddings = (
    word_embeddings
    + PE[tf.newaxis, :]
)

print("Word Embedding Shape:")
print(word_embeddings.shape)

print("\nPositional Encoding Shape:")
print(PE.shape)

print("\nPosition Aware Embeddings Shape:")
print(position_aware_embeddings.shape)

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    Dense,
    GlobalAveragePooling1D,
    Dropout,
    Layer
)
from tensorflow.keras.models import Model

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load Dataset

df = pd.read_csv("Resume.csv")

resume_col = None

for col in df.columns:
    if "resume" in col.lower():
        resume_col = col
        break

# Text Cleaning

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df["clean_resume"] = df[resume_col].apply(clean_text)

# Tokenization

tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(df["clean_resume"])

sequences = tokenizer.texts_to_sequences(
    df["clean_resume"]
)

# Vocabulary

vocab_size = len(tokenizer.word_index) + 1

# Padding

max_length = 300

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Labels

encoder = LabelEncoder()

y = encoder.fit_transform(
    df["Category"]
)

num_classes = len(np.unique(y))

# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Positional Encoding Function

def positional_encoding(position, d_model):

    positions = np.arange(position)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])

    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(PE, dtype=tf.float32)

# Custom Positional Encoding Layer

class PositionalEncoding(Layer):

    def __init__(self, max_position, d_model):

        super().__init__()

        self.pos_encoding = positional_encoding(
            max_position,
            d_model
        )

    def call(self, inputs):

        return inputs + self.pos_encoding[:tf.shape(inputs)[1], :]

# Parameters

embedding_dim = 64
num_heads = 2

# Input

inputs = Input(shape=(max_length,))

# Embedding

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)(inputs)

# Positional Encoding

position_encoded = PositionalEncoding(
    max_length,
    embedding_dim
)(embedding_layer)

# Multi Head Attention

attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=32
)(
    query=position_encoded,
    value=position_encoded,
    key=position_encoded
)

# Pooling

pooling = GlobalAveragePooling1D()(
    attention_output
)

# Dense

dense = Dense(
    64,
    activation='relu'
)(pooling)

drop = Dropout(0.3)(
    dense
)

# Output

outputs = Dense(
    num_classes,
    activation='softmax'
)(
    drop
)

# Model

improved_model = Model(
    inputs=inputs,
    outputs=outputs
)

improved_model.summary()

# Compile

improved_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train

start_time = time.time()

history_improved = improved_model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=8,
    verbose=1
)

end_time = time.time()

improved_training_time = (
    end_time - start_time
)

# Evaluate

improved_loss, improved_accuracy = (
    improved_model.evaluate(
        X_test,
        y_test,
        verbose=0
    )
)

# Results

print("\nImproved Model Results")

print(
    "Accuracy :",
    round(improved_accuracy * 100, 2),
    "%"
)

print(
    "Loss :",
    round(improved_loss, 4)
)

print(
    "Training Time :",
    round(improved_training_time, 2),
    "seconds"
)

# Save Metrics

improved_model_accuracy = improved_accuracy

improved_model_loss = improved_loss

improved_model_training_time = improved_training_time

In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Loss",
        "Training Time (sec)"
    ],

    "Without PE": [
        round(baseline_accuracy * 100, 2),
        round(baseline_loss, 4),
        round(baseline_training_time, 2)
    ],

    "With PE": [
        round(improved_model_accuracy * 100, 2),
        round(improved_model_loss, 4),
        round(improved_model_training_time, 2)
    ]

})

print(comparison)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["Accuracy", "Loss"]

without_pe = [
    baseline_accuracy * 100,
    baseline_loss
]

with_pe = [
    improved_model_accuracy * 100,
    improved_model_loss
]

x = np.arange(len(metrics))

width = 0.35

plt.figure(figsize=(8,5))

plt.bar(
    x - width/2,
    without_pe,
    width,
    label="Without PE"
)

plt.bar(
    x + width/2,
    with_pe,
    width,
    label="With PE"
)

plt.xticks(x, metrics)

plt.ylabel("Value")

plt.title(
    "Baseline vs Positional Encoding Model"
)

plt.legend()

plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

# Example Sentences

sentence1 = "Python developer with 5 years experience in AI"

sentence2 = "AI experience with Python developer 5 years"

print("Sentence 1:")
print(sentence1)

print("\nSentence 2:")
print(sentence2)

# Tokenization

tokenizer = Tokenizer()

tokenizer.fit_on_texts([
    sentence1,
    sentence2
])

seq1 = tokenizer.texts_to_sequences(
    [sentence1]
)[0]

seq2 = tokenizer.texts_to_sequences(
    [sentence2]
)[0]

print("\nToken IDs Sentence 1:")
print(seq1)

print("\nToken IDs Sentence 2:")
print(seq2)

# Positional Encoding Function

def positional_encoding(position, d_model):

    positions = np.arange(position)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])

    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(
        PE,
        dtype=tf.float32
    )

# Create Positional Encoding

max_position = max(
    len(seq1),
    len(seq2)
)

d_model = 8

PE = positional_encoding(
    max_position,
    d_model
)

print("\nPositional Encoding Shape:")
print(PE.shape)

# Display Positions

print("\nSentence 1 Positions")

for i, word in enumerate(sentence1.split()):

    print(
        f"Word: {word:12} Position: {i}"
    )

print("\nSentence 2 Positions")

for i, word in enumerate(sentence2.split()):

    print(
        f"Word: {word:12} Position: {i}"
    )

# Compare Same Word

print("\nWord Position Comparison")

print("Python in Sentence 1 -> Position",
      sentence1.split().index("Python"))

print("Python in Sentence 2 -> Position",
      sentence2.split().index("Python"))

print("\nAI in Sentence 1 -> Position",
      sentence1.split().index("AI"))

print("AI in Sentence 2 -> Position",
      sentence2.split().index("AI"))

# Show Positional Encoding Vectors

python_pos_s1 = sentence1.split().index("Python")
python_pos_s2 = sentence2.split().index("Python")

print("\nPython Positional Encoding in Sentence 1")

print(
    PE[python_pos_s1].numpy()
)

print("\nPython Positional Encoding in Sentence 2")

print(
    PE[python_pos_s2].numpy()
)

ai_pos_s1 = sentence1.split().index("AI")
ai_pos_s2 = sentence2.split().index("AI")

print("\nAI Positional Encoding in Sentence 1")

print(
    PE[ai_pos_s1].numpy()
)

print("\nAI Positional Encoding in Sentence 2")

print(
    PE[ai_pos_s2].numpy()
)

## Visualize Positional Encoding

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def positional_encoding(position, d_model):

    positions = np.arange(position)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])

    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(PE, dtype=tf.float32)

# Create Positional Encoding Matrix

max_position = 50
d_model = 64

PE = positional_encoding(
    max_position,
    d_model
)

print("Positional Encoding Shape:")
print(PE.shape)

# Heatmap Visualization

plt.figure(figsize=(12,6))

plt.imshow(
    PE.numpy(),
    aspect='auto',
    cmap='viridis'
)

plt.colorbar()

plt.xlabel("Embedding Dimensions")

plt.ylabel("Word Positions")

plt.title("Positional Encoding Heatmap")

plt.show()

In [ ]:
import pickle

# Save Model

improved_model.save(
    "model.keras"
)

print("Model Saved Successfully")

# Save Tokenizer

with open(
    "tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )

print("Tokenizer Saved Successfully")

# Save Label Encoder

with open(
    "label_encoder.pkl",
    "wb"
) as file:

    pickle.dump(
        encoder,
        file
    )

print("Label Encoder Saved Successfully")